In [ ]:
import torch

w = torch.rand(5, 5)
m = torch.rand(5, 5)

tril_mask = torch.tril(m, diagonal=-1)
triu_mask = torch.triu(m, diagonal=1)

w_masked = w.masked_fill(tril_mask.bool(), -torch.inf)

s = torch.softmax(w_masked, dim=-1)

print(tril_mask.bool())
print(triu_mask.bool())
print(w_masked)
print(s)

In [ ]:
import torch

x = torch.ones(5,5)
y = torch.triu(x, diagonal=-2)

print(y)

In [ ]:
import torch

x = torch.zeros(5,5)
mask = torch.eye(5,5).bool()

print(x)
print(mask)

x.masked_fill_(mask, 42)

print(x)


In [ ]:
torch.manual_seed(123)
dropout = torch.nn.Dropout(0.75)
example = torch.ones(6,6)
print(dropout(example))

In [ ]:
import torch
import torch.nn as nn

class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout= nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        num_tokens = x.shape[1]
        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1,2)
        attn_scores = torch.masked_fill(attn_scores,
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf
        )
        attn_weights = torch.softmax(attn_scores/ keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vec = attn_weights @ values
        return context_vec


d_in = 6
d_out = 9
context_length = 100
batch = torch.rand((2, 7, 6))
print(batch.shape)
ca = CausalAttention(d_in, d_out, context_length, 0.0)
print(ca(batch).shape)


In [ ]:
import torch
import torch.nn as nn

class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert(d_out % num_heads == 0), "d_out should be a multiple of num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)

        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        # b, num_tokens, d_out
        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)

        # b, num_tokens, num_heads, head_dim
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)

        # b, num_heads, num_tokens, head_dim
        queries = queries.transpose(1, 2)
        keys = keys.transpose(1, 2)
        values = values.transpose(1, 2)

        # b, num_heads, num_tokens, num_tokens
        attn_scores = queries @ keys.transpose(2,3)
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)

        # b, num_heads, num_tokens, head_dim
        context_vec = attn_weights @ values
        
        # b, num_tokens, num_heads, head_dim
        context_vec = context_vec.transpose(1 ,2)

        # b, num_tokens, d_out
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec)

        return context_vec



In [ ]:
import torch
from mha import MultiHeadAttention

In [ ]:
import torch
imporet torch.nn  as nn

class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert(d_out % num_heads == 0), "d_out should be a multiple of num_heads."

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value= nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.out_proj = nn.Linear(d_out, d_out)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, token_num, d_in = x.shape

        # b, token_num, d_out
        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)

        # b, token_num, num_heads, head_dim
        queries = queries.view(b, token_num, self.num_heads, self.head_dim)
        keys = keys.view(b, token_num, self.num_heads, self.head_dim)
        values = values.view(b, token_num, self.num_heads, self.head_dim)

        # b, num_heads, token_num, head_dim
        queries = queries.transpose(1, 2)
        keys = keys.transpose(1, 2)
        values= values.transpose(1, 2)

        # b, num_heads, token_num, token_num
        attn_scores = queries @ keys.transpose(2, 3)
        mask_bool = self.mask.bool()[:token_num, :token_num]
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)

        # b, num_heads, token_num, head_dim
        context_vec = attn_weights @ values

        # b, token_num, num_heads, head_dim
        context_vec = context_vec.transpose(1, 2)

        # b, token_num, d_out
        context_vec = context_vec.contiguous().view(b, token_num, self.d_out)
        context_vec = self.out_proj(context_vec)

        return context_vec



In [ ]:
inputs = torch.tensor(
    [[0.43, 0.15, 0.89],
     [0.55, 0.87, 0.66],
     [0.57, 0.85, 0.64],
     [0.22, 0.58, 0.22],
     [0.77, 0.25, 0.10],
     [0.05, 0.80, 0.55]]
)

batch = torch.stack((inputs, inputs), dim=0)
print(batch.shape)

torch.manual_seed(123)
context_length = batch.shape[1]
d_in, d_out = 3, 6
mha = MultiHeadAttention(
    d_in, d_out, context_length, 0.0, num_heads=3
)
context_vec = mha(batch)

#print(context_vec)
print(context_vec.shape)





In [ ]:
import torch

x = torch.arange(4).view(2,2)

a = x.reshape(4)
a[0] = 99
print(x[0,0])

In [ ]:
y = x.t()
b = y.reshape(4)
b[0] = -1
print(y[0,0])

In [ ]:
print(b)

In [ ]:
import torch
torch.cuda.is_available()

In [ ]:
import torch
import torch.nn as nn

class DeviceTestModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.plain_tensor = torch.ones(2,2)
        self.register_buffer('registered_buffer', torch.ones(2,2))

model = DeviceTestModel()

model.to('cuda')

print(model.plain_tensor.device)
print(model.registered_buffer.device)